# ALS (Alternating Least Squares) разложение

Необходимо по истории взаимодействий пользователя предсказать товары, которые он закажет в ближайшие 2 недели.

Для подбора гиперпараметров из обучающей выборки последние 2 недели выделены под валидацию (с 18 июня 2025 по 1 июля 2025)

Было сформировано 2 варианта матрицы `users × items`:
- на всех доступных данных (с 1 января 2025 по 17 июня 2025)
- на данных за последние два месяца (с 18 апреля 2025 по 17 июня 2025)

Действия пользователя разделены на группы:
- **Order** - (явная покупка, максимальный вес в матрице)
- **Strong** - явные действия намерения, но не гарантия покупки (добавление в корзину / избранное)
- **Weak** - исследовательское поведение (просмотр товара, опичание, отзывов) - накапливается до определенного порога

Полученные метрики:
- NDCG@100 on val: 0.00997
- NDCG@100 on test: 0.01183

Примечания:
- сокращение окна обучения с 6,5 до 2 месяцев существенно улучшило метрику
- увеличение factors с 64 до 128 существенно улучшило метрику (на 256 ядро падает)
- усиление регуляризации незначительно ухудшило метрику
- применение bm25_weight незначительно ухудшило метрику

Зона для улучшения:
- изменение весов событий (очень долго, нужен пересбор матрицы `users × items`)
- ослабление week событий со временем (очень долго, нужен пересбор матрицы `users × items`)

In [1]:
import os
import sys
from collections import defaultdict
from datetime import date, timedelta
from functools import reduce
from pathlib import Path
from tqdm import trange

import dask.dataframe as dd
import implicit
import numpy as np
import pandas as pd
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix, save_npz, load_npz

sys.path.append('../src')

from data_loader import (
    download_dataset,
    load_orders,
    load_interactions,
    load_items
)

os.environ["OPENBLAS_NUM_THREADS"] = "1"

In [2]:
download_dataset()

Path("../models").mkdir(parents=True, exist_ok=True)
Path("../data/interim").mkdir(parents=True, exist_ok=True)

Датасет загружен ранее


In [3]:
TRAIN_START = date(2025, 4, 18)
TRAIN_END  = date(2025, 6, 17)

VAL_START  = date(2025, 6, 18)
VAL_END    = date(2025, 7, 1)

ACTIVE_ITEMS_DAYS = 3
WEAK_EVENTS   = {"page_view", "view_description", "review_view"}
STRONG_EVENTS = {"to_cart", "favorite"}

# Веса событий (WEAK_EVENTS накапливаются до WEAK_MAX_CAP)
EVENT_WEIGHTS = {
    "page_view": 1,
    "view_description": 2,
    "review_view": 3,
    "favorite": 20,
    "to_cart": 100,
    "order": 100,
}

# Ограничение кумулятивного веса для WEAK_EVENTS
WEAK_MAX_CAP = 20

# ---------------------------
# Time decay (затухание веса по давности события)
# ---------------------------
# Если включено, веса событий умножаются на decay(age_days),
# где age_days = (TRAIN_END - event_date).days
TIME_DECAY_ENABLED = True

# "exp"  : exp(-age / tau)
# "piecewise": ступенчатые коэффициенты по бинам
TIME_DECAY_MODE = "exp"

# Для exp
TIME_DECAY_TAU_DAYS = 14          # попробуйте 7 / 14 / 30
TIME_DECAY_MIN_FACTOR = 0.10      # нижняя граница, чтобы старые события не обнулялись

# Для piecewise (длина FACTORS = len(BINS)+1)
# age <= 3  -> 1.0
# 4..14     -> 0.7
# 15..30    -> 0.4
# 31..60    -> 0.2
# 61+       -> 0.1
TIME_DECAY_BINS_DAYS = [3, 14, 30, 60]
TIME_DECAY_FACTORS  = [1.0, 0.7, 0.4, 0.2, 0.1]

# Количество шардов для сохранения матрицы users × items
N_SHARDS = 128    # если упадёт — 256  # на 64 упало на 6 месяцах

# Директория для сохранения шардов
OUT_DIR = f"../data/processed/csr_build_{TRAIN_START}_{TRAIN_END}"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

## Формироване матрицы **users × items**

In [4]:
def filter_tracker_ddf(
    tracker_ddf: dd.DataFrame,
    start_date: date,
    end_date: date,
    keep_date: bool = False,
) -> dd.DataFrame:
    """
    Фильтрация действий пользователей по времени
    """

    tracker_ddf = tracker_ddf.assign(
        date=dd.to_datetime(tracker_ddf["date"], format="%Y-%m-%d")
    )

    filtered = tracker_ddf[
        (tracker_ddf["date"] >= pd.to_datetime(start_date)) &
        (tracker_ddf["date"] <= pd.to_datetime(end_date)) &
        (tracker_ddf["action_type"].isin(WEAK_EVENTS | STRONG_EVENTS))
    ]#[["user_id", "item_id", "action_type"]]

    cols = ["user_id", "item_id", "action_type"]
    if keep_date:
        cols.append("date")

    return filtered[cols]

In [5]:
def filter_orders_ddf(
    orders_ddf: dd.DataFrame,
    start_date: date,
    end_date: date,
    keep_created_date: bool = False,
) -> dd.DataFrame:
    """
    Фильтрация заказов пользователей пользователя по времени
    """

    orders_ddf = orders_ddf.assign(
        created_date=dd.to_datetime(orders_ddf["created_date"], format="%Y-%m-%d")
    )

    filtered = orders_ddf[
        (orders_ddf["created_date"] >= pd.to_datetime(start_date)) &
        (orders_ddf["created_date"] <= pd.to_datetime(end_date))
    ]#[["user_id", "item_id"]]

    cols = ["user_id", "item_id"]
    if keep_created_date:
        cols.append("created_date")
    return filtered[cols]


In [6]:
def build_id_mappings(
    tracker_ddf: dd.DataFrame,
    orders_ddf: dd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Формирование маппингов реальных id пользователя/товара на idx для ALS
    """

    users = dd.concat([
        tracker_ddf[["user_id"]],
        orders_ddf[["user_id"]],
    ]).drop_duplicates().compute()

    items = dd.concat([
        tracker_ddf[["item_id"]],
        orders_ddf[["item_id"]],
    ]).drop_duplicates().compute()

    users = users["user_id"].astype("int64").sort_values().reset_index(drop=True)
    items = items["item_id"].astype("int64").sort_values().reset_index(drop=True)

    user_mapping = pd.DataFrame({
        "user_id": users,
        "user_idx": np.arange(len(users), dtype=np.int32),
    })

    item_mapping = pd.DataFrame({
        "item_id": items,
        "item_idx": np.arange(len(items), dtype=np.int32),
    })

    return user_mapping, item_mapping


In [7]:
def _decay_factor_from_age(age_days: np.ndarray) -> np.ndarray:
    """
    age_days: float32 array (>= 0)
    returns: float32 array
    """
    if TIME_DECAY_MODE == "exp":
        tau = float(TIME_DECAY_TAU_DAYS)
        f = np.exp(-age_days / tau).astype(np.float32)
        if TIME_DECAY_MIN_FACTOR is not None:
            f = np.maximum(f, float(TIME_DECAY_MIN_FACTOR)).astype(np.float32)
        return f
    elif TIME_DECAY_MODE == "piecewise":
        bins = np.asarray(TIME_DECAY_BINS_DAYS, dtype=np.float32)
        factors = np.asarray(TIME_DECAY_FACTORS, dtype=np.float32)
        if len(factors) != len(bins) + 1:
            raise ValueError("TIME_DECAY_FACTORS must have len(TIME_DECAY_BINS_DAYS) + 1")
        idx = np.digitize(age_days, bins, right=True)  # 0..len(bins)
        return factors[idx].astype(np.float32)
    else:
        raise ValueError(f"Unknown TIME_DECAY_MODE={TIME_DECAY_MODE}")

In [8]:
def build_interactions_ddf(
    tracker_pdf: pd.DataFrame,
    orders_pdf: pd.DataFrame,
    ref_date: pd.Timestamp | None = None,
) -> pd.DataFrame:
    """
    Формирование датафрейма user-item-weight
    """

    tracker_pdf["weight"] = (
        tracker_pdf["action_type"]
        .map(EVENT_WEIGHTS)
        .astype(np.float32)
    )

    # time decay для трекера
    if TIME_DECAY_ENABLED:
        if ref_date is None:
            raise ValueError("ref_date must be provided when TIME_DECAY_ENABLED=True")
        if "date" not in tracker_pdf.columns:
            raise ValueError("tracker_pdf must contain 'date' when TIME_DECAY_ENABLED=True")
        tracker_pdf["date"] = pd.to_datetime(tracker_pdf["date"])
        age_days = (ref_date - tracker_pdf["date"]).dt.days.clip(lower=0).astype(np.float32).values
        tracker_pdf["weight"] *= _decay_factor_from_age(age_days)

    weak = (
        tracker_pdf[tracker_pdf["action_type"].isin(WEAK_EVENTS)]
        .groupby(["user_id", "item_id"], as_index=False)
        .agg(weight=("weight", "sum"))
    )
    weak["weight"] = weak["weight"].clip(upper=float(WEAK_MAX_CAP)).astype(np.float32)

    strong = (
        tracker_pdf[tracker_pdf["action_type"].isin(STRONG_EVENTS)]
        .groupby(["user_id", "item_id"], as_index=False)
        .agg(weight=("weight", "max"))
    )

    # orders: если есть created_date, применяем time decay и берём max (самый свежий заказ даст самый большой вес)
    if TIME_DECAY_ENABLED:
        if ref_date is None:
            raise ValueError("ref_date must be provided when TIME_DECAY_ENABLED=True")
        if "created_date" not in orders_pdf.columns:
            raise ValueError("orders_pdf must contain 'created_date' when TIME_DECAY_ENABLED=True")
        orders_pdf["created_date"] = pd.to_datetime(orders_pdf["created_date"])
        age_days_o = (ref_date - orders_pdf["created_date"]).dt.days.clip(lower=0).astype(np.float32).values
        orders_pdf = orders_pdf.assign(
            weight=(float(EVENT_WEIGHTS["order"]) * _decay_factor_from_age(age_days_o)).astype(np.float32)
        )
        orders = (
            orders_pdf
            .groupby(["user_id", "item_id"], as_index=False)
            .agg(weight=("weight", "max"))
        )
    else:
        orders = (
            orders_pdf
            .drop_duplicates()
            .assign(weight=float(EVENT_WEIGHTS["order"]))
            .astype({"weight": np.float32})
        )

    all_events = pd.concat([weak, strong, orders], ignore_index=True)

    interactions = (
        all_events
        .groupby(["user_id", "item_id"], as_index=False)
        .agg(weight=("weight", "max"))
    )

    interactions["weight"] = interactions["weight"].astype(np.float32)
    return interactions.astype({
        "user_id": "int32",
        "item_id": "int32",
    })


In [9]:
def process_shard(
    shard_id: int,
    n_shards: int,
    tracker_ddf: dd.DataFrame,
    orders_ddf: dd.DataFrame,
    u2i: dict,
    i2i: dict,
    n_users: int,
    n_items: int,
    out_dir: str,
    ref_date: pd.Timestamp | None = None,
):
    """
    Формирование матрицы взаимодействий user-item для шарда
    """

    tracker_pdf = tracker_ddf[
        (tracker_ddf["user_id"] % n_shards) == shard_id
    ].compute()

    orders_pdf = orders_ddf[
        (orders_ddf["user_id"] % n_shards) == shard_id
    ].compute()

    if tracker_pdf.empty and orders_pdf.empty:
        return

    interactions = build_interactions_ddf(tracker_pdf, orders_pdf, ref_date=ref_date)

    interactions["u"] = interactions["user_id"].map(u2i)
    interactions["i"] = interactions["item_id"].map(i2i)

    interactions.dropna(subset=["u", "i"], inplace=True)

    mat = csr_matrix(
        (
            interactions["weight"].astype(np.float32).values,
            (interactions["u"].values.astype("int32"),
             interactions["i"].values.astype("int32")),
        ),
        shape=(n_users, n_items),
    )

    save_npz(f"{out_dir}/shard_{shard_id}.npz", mat)


In [10]:
def build_interaction_csr(
    tracker_ddf: dd.DataFrame,
    orders_ddf: dd.DataFrame,
    start_date,
    end_date,
    n_shards: int,
    out_dir: str,
    active_items_days: int | None = None,
):
    """
    Формирование матриц взаимодействий user-item по шардам
    и сохранение маппингов
    """
    need_recency_cols = (active_items_days is not None) or TIME_DECAY_ENABLED

    tracker_ddf = filter_tracker_ddf(
        tracker_ddf, start_date, end_date, keep_date=need_recency_cols
    )
    orders_ddf = filter_orders_ddf(
        orders_ddf, start_date, end_date, keep_created_date=need_recency_cols
    )

    # Фильтруем items: оставляем только те, у которых были события за последние N дней
    # (внутри [start_date, end_date], относительно end_date).
    if active_items_days is not None:
        if active_items_days <= 0:
            raise ValueError("active_items_days must be > 0")

        active_start = end_date - timedelta(days=active_items_days - 1)
        active_start_ts = pd.to_datetime(active_start)

        recent_items = dd.concat(
            [
                tracker_ddf[tracker_ddf["date"] >= active_start_ts][["item_id"]],
                orders_ddf[orders_ddf["created_date"] >= active_start_ts][["item_id"]],
            ],
            axis=0,
        ).drop_duplicates()

        # Оставляем в полном периоде только активные item_id (по списку из "последних N дней")
        tracker_ddf = tracker_ddf.merge(recent_items, on="item_id", how="inner")
        orders_ddf = orders_ddf.merge(recent_items, on="item_id", how="inner")

        # Дальше даты не нужны только если НЕ используем time-decay
        if not TIME_DECAY_ENABLED:
            tracker_ddf = tracker_ddf[["user_id", "item_id", "action_type"]]
            orders_ddf = orders_ddf[["user_id", "item_id"]]

    user_mapping, item_mapping = build_id_mappings(
        tracker_ddf, orders_ddf
    )

    u2i = dict(zip(user_mapping["user_id"], user_mapping["user_idx"]))
    i2i = dict(zip(item_mapping["item_id"], item_mapping["item_idx"]))

    n_users = len(user_mapping)
    n_items = len(item_mapping)

    user_mapping.to_parquet(f"{out_dir}/user_mapping.parquet", index=False)
    item_mapping.to_parquet(f"{out_dir}/item_mapping.parquet", index=False)

    ref_ts = pd.to_datetime(end_date)
    for shard_id in range(n_shards):
        print(f"Shard {shard_id + 1}/{n_shards}")
        process_shard(
            shard_id,
            n_shards,
            tracker_ddf,
            orders_ddf,
            u2i,
            i2i,
            n_users,
            n_items,
            out_dir,
            ref_date=ref_ts,
        )

In [11]:
def build_final_csr(out_dir: str, n_shards: int):
    """
    Сборка csr матрицы user-item из сохраненных ранее шардов
    """
    mats = []
    for k in range(n_shards):
        try:
            mats.append(load_npz(f"{out_dir}/shard_{k}.npz"))
        except FileNotFoundError:
            pass

    final = reduce(lambda a, b: a + b, mats)
    save_npz(f"{out_dir}/interaction_matrix.npz", final)
    return final

In [ ]:
tracker_ddf = load_interactions(train=True)
orders_ddf = load_orders(train=True)

build_interaction_csr(
    tracker_ddf=tracker_ddf,
    orders_ddf=orders_ddf,
    start_date=NEW_TRAIN_START,
    end_date=NEW_TRAIN_END,
    n_shards=N_SHARDS,
    out_dir=OUT_DIR,
    active_items_days=ACTIVE_ITEMS_DAYS,
)

Shard 1/128
Shard 2/128
Shard 3/128
Shard 4/128
Shard 5/128
Shard 6/128
Shard 7/128
Shard 8/128
Shard 9/128
Shard 10/128
Shard 11/128
Shard 12/128
Shard 13/128
Shard 14/128
Shard 15/128
Shard 16/128
Shard 17/128
Shard 18/128
Shard 19/128
Shard 20/128
Shard 21/128
Shard 22/128
Shard 23/128
Shard 24/128
Shard 25/128
Shard 26/128
Shard 27/128
Shard 28/128
Shard 29/128
Shard 30/128
Shard 31/128
Shard 32/128
Shard 33/128
Shard 34/128
Shard 35/128
Shard 36/128
Shard 37/128
Shard 38/128
Shard 39/128
Shard 40/128
Shard 41/128
Shard 42/128
Shard 43/128
Shard 44/128
Shard 45/128
Shard 46/128
Shard 47/128
Shard 48/128
Shard 49/128
Shard 50/128
Shard 51/128
Shard 52/128
Shard 53/128
Shard 54/128
Shard 55/128
Shard 56/128
Shard 57/128
Shard 58/128
Shard 59/128
Shard 60/128
Shard 61/128
Shard 62/128
Shard 63/128
Shard 64/128
Shard 65/128
Shard 66/128
Shard 67/128
Shard 68/128
Shard 69/128
Shard 70/128
Shard 71/128
Shard 72/128
Shard 73/128
Shard 74/128
Shard 75/128
Shard 76/128
Shard 77/128
Shard 78

In [13]:
user_items_csr = build_final_csr(
    out_dir=OUT_DIR,
    n_shards=N_SHARDS,
)

user_items_csr

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 235637945 stored elements and shape (951085, 2174580)>

## Обучение ALS

In [14]:
user_items_csr = load_npz(f"{OUT_DIR}/interaction_matrix.npz")
user_items_csr    # users × items

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 235637945 stored elements and shape (951085, 2174580)>

In [15]:
model = AlternatingLeastSquares(
    factors=128,
    regularization=0.01,
    iterations=20,
    use_gpu=False,
)

model.fit(user_items_csr)

  0%|          | 0/20 [00:00<?, ?it/s]

In [16]:
model.save("../models/als_model_for_val.npz")

user_map = pd.read_parquet(f"{OUT_DIR}/user_mapping.parquet")
item_map = pd.read_parquet(f"{OUT_DIR}/item_mapping.parquet")

user_map.to_parquet("../models/als_val_user_mapping.parquet", index=False)
item_map.to_parquet("../models/als_val_item_mapping.parquet", index=False)

print("Сохранено:")
print(" - ../models/als_model_for_val.npz")
print(" - ../models/als_val_user_mapping.parquet")
print(" - ../models/als_val_item_mapping.parquet")

Сохранено:
 - ../models/als_model_for_val.npz
 - ../models/als_val_user_mapping.parquet
 - ../models/als_val_item_mapping.parquet


In [17]:
assert user_items_csr.shape[0] == model.user_factors.shape[0]
assert user_items_csr.shape[1] == model.item_factors.shape[0]

In [18]:
orders_ddf = load_orders(train=True)
orders_ddf.head()

,item_id,user_id,created_timestamp,last_status,last_status_timestamp,created_date
0,15535,3423970,2025-01-11 19:20:02.550,delivered_orders,2025-01-16 10:53:03,2025-01-11
1,15535,1207601,2025-03-25 13:11:48.240,canceled_orders,2025-03-29 08:32:43,2025-03-25
2,119241,2340021,2025-05-03 20:46:18.146,canceled_orders,2025-05-12 17:10:27,2025-05-03
3,119241,2356390,2025-05-01 17:53:00.716,delivered_orders,2025-05-14 14:36:20,2025-05-01
4,119241,1490080,2025-04-22 22:59:11.160,canceled_orders,2025-04-26 06:27:28,2025-04-22


In [19]:
user_map = pd.read_parquet(f"{OUT_DIR}/user_mapping.parquet")

user_id = 3423970
u_idx = user_map.loc[user_map.user_id == user_id, "user_idx"].iloc[0]
u_idx

np.int32(643388)

In [20]:
items, scores = model.recommend(
    userid=u_idx,
    user_items=user_items_csr[u_idx],
    N=100,
    filter_already_liked_items=True,
)
items

array([1855184, 1577672,  699259, 1167988, 1003076,  959894, 2061058,
         57396,  374198, 1396414, 1278065,    1562,  566832,  188093,
       1792125,  213699,  282546,  924549,  295998,  473499,   40175,
        263244,  175548, 1326364,  629614, 2042396, 1358275,  300573,
       1700296, 1108842,  485620,    8757,  891473,  657457, 1703401,
       2010860,  914521,  217545, 1549115, 1525180, 1987295,  556027,
        100611,  184786,   74641,  598647, 2029528,  995746,  384620,
        224779, 2005610, 1922469, 1807177, 1649956, 2052583,  479498,
       1012587,  884202,  917599,  467443, 1226311,  635976,  379931,
        417828,  522970,  532241, 2159510,  181399, 1220998, 2064094,
       1257858,  242896, 1052198, 1431327, 1415284, 1137591, 1156141,
        299580,  730202, 1179349, 1727270, 2011756,  871405,  856789,
       1329388, 1499263, 1070681, 1820093,  153589, 1870567,  892623,
       1105663, 2048379,  937523,  990710,  656358, 1812936,  810073,
        966833,  149

In [21]:
item_map = pd.read_parquet(f"{OUT_DIR}/item_mapping.parquet")
items_ddf = load_items()

item_id = item_map[item_map['item_idx'].isin(items)]['item_id']

recs = items_ddf[items_ddf['item_id'].isin(item_id)].compute()
recs

,item_id,itemname,attributes,fclip_embed,catalogid,variant_id,model_id
61894,58318760,Сарафан Harper's Bazaar (журнал),"[{'attribute_name': 'Type', 'attribute_value':...","[-0.05972548, 1.671419, 0.86633307, -0.1296886...",36741,24984187,37023602
6288,320377395,Сарафан Babelini,"[{'attribute_name': 'Type', 'attribute_value':...","[-0.015324801, 1.3843509, -0.18187198, 0.47419...",7502,306632780,33775138
42654,44053675,Платье KVARTALL Лето,"[{'attribute_name': 'Type', 'attribute_value':...","[0.065433115, 1.0824628, -0.03450963, 0.166328...",7502,156989405,177490
63603,137738567,Платье Style World,"[{'attribute_name': 'Type', 'attribute_value':...","[0.15117484, 1.2063283, 0.13268386, -0.4459535...",7502,98545059,26413755
29788,33272673,Платье,"[{'attribute_name': 'Type', 'attribute_value':...","[0.3233093, 2.4415207, 0.271424, -0.18755977, ...",7502,101459619,31804840
...,...,...,...,...,...,...,...
11549,27343755,Юбка,"[{'attribute_name': 'Type', 'attribute_value':...","[-0.09988938, 2.0802026, -0.57043815, -0.24365...",7504,285862886,14118205
23416,142962486,Платье,"[{'attribute_name': 'Type', 'attribute_value':...","[0.21220262, 2.2300916, 0.23739964, -0.4735547...",36741,141405724,31804840
9646,144047073,Платье,"[{'attribute_name': 'Type', 'attribute_value':...","[-0.46878576, 1.4568192, -0.32564217, 0.012762...",7502,45959679,29387635
28251,237179,Футболка Апрель,"[{'attribute_name': 'Type', 'attribute_value':...","[-0.18020874, 1.0694798, -0.17670688, 0.144358...",7508,138219923,14882893


## Оценка NDCG@100

In [22]:
orders_ddf = load_orders(train=True)
user_items_csr = load_npz(f"{OUT_DIR}/interaction_matrix.npz")
model = implicit.cpu.als.AlternatingLeastSquares.load("../models/als_model_for_val.npz")

### Подготовка ground truth для валидации (реальные покупки пользователей)

In [ ]:
def make_df_for_eval(orders_ddf_eval, out_dir, start_date, end_date):
    """
    Получение выборки для оценки ALS:
    - клиент делал заказы в период оценки
    - клиент присутствует в обучающей выборке

    ВАЖНО:
    - пользователей фильтруем до warm users
    - товары из GT НЕ выкидываем, даже если их нет в item_mapping
      (такие товары остаются в релевантных как guaranteed misses)
    """
    user_map = pd.read_parquet(f"{out_dir}/user_mapping.parquet")
    item_map = pd.read_parquet(f"{out_dir}/item_mapping.parquet")

    user2idx = dict(zip(user_map.user_id, user_map.user_idx))
    item2idx = dict(zip(item_map.item_id, item_map.item_idx))

    orders_ddf_eval = orders_ddf_eval.assign(
        created_date=dd.to_datetime(orders_ddf_eval["created_date"])
    )

    eval_df = orders_ddf_eval[
        (orders_ddf_eval["created_date"] >= pd.to_datetime(start_date)) &
        (orders_ddf_eval["created_date"] <= pd.to_datetime(end_date))
    ][["user_id", "item_id"]].compute()

    # Оставляем только warm users
    eval_df = eval_df[eval_df["user_id"].isin(user2idx)].copy()

    eval_df["u"] = eval_df["user_id"].map(user2idx).astype(np.int32)

    # Для warm items используем item_idx,
    # для cold items сохраняем уникальный отрицательный surrogate id,
    # чтобы они остались в GT и гарантированно не матчились с рекомендациями ALS
    eval_df["i"] = eval_df["item_id"].map(item2idx)

    cold_mask = eval_df["i"].isna()
    if cold_mask.any():
        eval_df.loc[cold_mask, "i"] = -(
            eval_df.loc[cold_mask, "item_id"].astype("int64") + 1
        )

    eval_df["i"] = eval_df["i"].astype(np.int64)

    gt = defaultdict(set)
    for u, i in zip(eval_df["u"], eval_df["i"]):
        gt[int(u)].add(int(i))

    users_for_eval = eval_df["u"].unique().tolist()

    return eval_df, gt, users_for_eval

In [24]:
eval_df, gt, users_for_eval = make_df_for_eval(
    orders_ddf, OUT_DIR, VAL_START, VAL_END)

display(eval_df.head())
print("Users in val GT:", len(gt))
print("Avg GT size:", np.mean([len(v) for v in gt.values()]))

,user_id,item_id,u,i
9,2902640,191536,545309,1268
11,1111231,191536,208903,1268
15,898360,219368,168931,1431
26,1623550,320379,305076,2103
44,4075081,567620,765742,3693


Users in val GT: 339286
Avg GT size: 3.9323491096007497


### Ранее купленные товары пользавателями

In [25]:
def build_purchased_by_user_idx(
    orders_ddf: dd.DataFrame,
    user_map: pd.DataFrame,
    item_map: pd.DataFrame,
    start_date,
    end_date,
):
    """
    OUTPUT:
        dict[user_idx] -> np.array(item_idx, dtype=int32)
    """

    orders_ddf = orders_ddf.assign(
        created_date=dd.to_datetime(
            orders_ddf["created_date"], format="%Y-%m-%d"
        )
    )

    orders_ddf = orders_ddf[
        (orders_ddf["created_date"] >= pd.to_datetime(start_date)) &
        (orders_ddf["created_date"] <= pd.to_datetime(end_date))
    ][["user_id", "item_id"]]

    orders_ddf = orders_ddf.astype({
        "user_id": "int64",
        "item_id": "int64",
    })

    user_map = user_map.astype({"user_id": "int64"})
    item_map = item_map.astype({"item_id": "int64"})

    tmp = (
        orders_ddf
        .merge(user_map[["user_id", "user_idx"]], on="user_id", how="inner")
        .merge(item_map[["item_id", "item_idx"]], on="item_id", how="inner")
        [["user_idx", "item_idx"]]
        .drop_duplicates()
        .compute()
    )

    purchased = defaultdict(list)

    for u, i in zip(tmp.user_idx.values, tmp.item_idx.values):
        purchased[u].append(i)

    return {
        u: np.array(items, dtype=np.int32)
        for u, items in purchased.items()
    }

In [26]:
user_map = pd.read_parquet(f"{OUT_DIR}/user_mapping.parquet")
item_map = pd.read_parquet(f"{OUT_DIR}/item_mapping.parquet")

purchased_by_user = build_purchased_by_user_idx(
    orders_ddf, user_map, item_map, TRAIN_START, TRAIN_END)

purchased_by_user

{np.int32(790662): array([   1008, 2168200,  403809,  284168, 1395587, 1244093,  445408,
         236039,  207327, 1454569,  759613,  332451, 2027234,  286408,
         138634, 1800790, 1537517,   23408, 2173013,  811166,  638498,
         623823, 1189583, 1754471,  342317, 1234396, 1548106,  572467,
         676112, 2066765,  808116,  644705, 1199994,  436722, 1203105,
        2157430, 1982058,   49920, 1989850,  730501, 1968106,  816261,
        1664965,  426415, 1345017, 1777065, 2068448,  401532,  121221,
        1523611,  123564, 1994070, 1308457, 1392409,  784043, 1464901,
        1330230, 2005345,  665284,  514087, 1923919,  396483, 1916868,
         479205,  380166, 1034573, 1105851,  303343,  323021, 1977044,
        2082397,  105470, 2039086, 1851663, 1282620, 1571630, 1162160,
         772526, 1889030,  764182,  190735,  908436, 1699373,  574888,
        1174345,  810549, 1089713,  306043, 1319477,  348135,  592992,
         806699, 1480224,  677682, 1172276, 1961243,   4730

### Оценка работы модели

In [27]:
def batch_recommend_no_filter(
    model,
    matrix,
    user_idxs,
    k,
):
    """
    Возвращает raw top-k рекомендаций без фильтрации:
    - item_idxs: batch x k
    - item_scores: batch x k
    """
    user_items = matrix[user_idxs]

    item_idxs, item_scores = model.recommend(
        userid=user_idxs,
        user_items=user_items,
        N=k,
        filter_already_liked_items=False,
    )

    return item_idxs, item_scores

In [28]:
def postprocess_recommendations(
    batch_users,
    batch_items,
    batch_scores,
    purchased_by_user,
    k,
):
    """
    Исключение из рекомендаций ранее купленных товаров
    с сохранением score.
    """
    results = {}

    for u, recs, scores in zip(batch_users, batch_items, batch_scores):
        purchased = purchased_by_user.get(u)
        if purchased is not None:
            purchased = set(purchased)

            filtered_items = []
            filtered_scores = []

            for item_idx, score in zip(recs, scores):
                if item_idx not in purchased:
                    filtered_items.append(int(item_idx))
                    filtered_scores.append(float(score))

            recs = filtered_items
            scores = filtered_scores

        results[u] = (
            np.array(recs[:k], dtype=np.int32),
            np.array(scores[:k], dtype=np.float32),
        )

    return results

In [29]:
def build_candidates_df(
    model,
    matrix,
    user_idxs,
    purchased_by_user,
    user_map: pd.DataFrame,
    item_map: pd.DataFrame,
    k: int = 200,
    oversample: int = 1000,
    batch_size: int = 512,
) -> pd.DataFrame:
    """
    Строит таблицу ALS-кандидатов в исходных user_id / item_id:
    user_id | item_id | als_score | als_rank | in_als
    """
    idx2user = dict(zip(user_map["user_idx"], user_map["user_id"]))
    idx2item = dict(zip(item_map["item_idx"], item_map["item_id"]))

    rows = []

    for i in trange(0, len(user_idxs), batch_size):
        batch_users = np.array(user_idxs[i:i + batch_size], dtype=np.int32)

        batch_items, batch_scores = batch_recommend_no_filter(
            model=model,
            matrix=matrix,
            user_idxs=batch_users,
            k=oversample,
        )

        batch_recs = postprocess_recommendations(
            batch_users=batch_users,
            batch_items=batch_items,
            batch_scores=batch_scores,
            purchased_by_user=purchased_by_user,
            k=k,
        )

        for u_idx in batch_users:
            user_id = idx2user.get(int(u_idx))
            if user_id is None:
                continue

            item_idxs, item_scores = batch_recs[int(u_idx)]

            for rank, (item_idx, item_score) in enumerate(zip(item_idxs, item_scores), start=1):
                item_id = idx2item.get(int(item_idx))
                if item_id is None:
                    continue

                rows.append(
                    {
                        "user_id": int(user_id),
                        "item_id": int(item_id),
                        "als_score": float(item_score),
                        "als_rank": int(rank),
                        "in_als": 1,
                    }
                )

    return pd.DataFrame(rows)


def build_all_test_user_idxs(
    orders_ddf_test: dd.DataFrame,
    out_dir: str,
):
    """
    Берём всех test users, которые присутствуют в train-mapping.
    Это нужно для финального candidate generation.
    """
    user_map = pd.read_parquet(f"{out_dir}/user_mapping.parquet")
    user2idx = dict(zip(user_map.user_id, user_map.user_idx))

    test_users_pdf = orders_ddf_test[["user_id"]].drop_duplicates().compute()
    test_users_pdf = test_users_pdf[test_users_pdf["user_id"].isin(user2idx)].copy()
    test_users_pdf["u"] = test_users_pdf["user_id"].map(user2idx)

    return test_users_pdf["u"].astype(np.int32).tolist()

In [30]:
# ------------------------------------------------------------
# SAVE VAL CANDIDATES FOR SECOND LEVEL
# ------------------------------------------------------------
user_map = pd.read_parquet(f"{OUT_DIR}/user_mapping.parquet")
item_map = pd.read_parquet(f"{OUT_DIR}/item_mapping.parquet")

als_val_candidates = build_candidates_df(
    model=model,
    matrix=user_items_csr,
    user_idxs=users_for_eval,
    purchased_by_user=purchased_by_user,
    user_map=user_map,
    item_map=item_map,
    k=200,
    oversample=1000,
    batch_size=512,
)

als_val_candidates.to_parquet("../data/interim/als_val_candidates.parquet", index=False)

print("Сохранено:")
print(" - ../data/interim/als_val_candidates.parquet")
als_val_candidates.head()

100%|██████████| 663/663 [21:16<00:00,  1.93s/it]


Сохранено:
 - ../data/interim/als_val_candidates.parquet


,user_id,item_id,als_score,als_rank,in_als
0,2902640,51974017,0.831209,1,1
1,2902640,166836286,0.401483,2,1
2,2902640,93305898,0.363785,3,1
3,2902640,77696741,0.362501,4,1
4,2902640,8952890,0.357628,5,1


In [31]:
def ndcg_at_k(recommended, relevant, k):
    if not relevant:
        return 0.0

    dcg = 0.0
    for rank, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            dcg += 1.0 / np.log2(rank + 1)

    idcg = sum(
        1.0 / np.log2(i + 1)
        for i in range(1, min(len(relevant), k) + 1)
    )

    return dcg / idcg if idcg > 0 else 0.0


In [32]:
def evaluate_ndcg(
    model,
    matrix,
    users_for_eval,
    gt,
    purchased_by_user,
    k=100,
    batch_size=512,
    oversample=1000
):
    scores = []

    for i in trange(0, len(users_for_eval), batch_size):
        batch_users = np.array(
            users_for_eval[i:i + batch_size],
            dtype=np.int32,
        )

        # Рекомендации с запасом
        batch_items, batch_scores = batch_recommend_no_filter(
            model,
            matrix,
            batch_users,
            k=oversample,
        )

        # Исключение купленных товаров
        batch_recs = postprocess_recommendations(
            batch_users=batch_users,
            batch_items=batch_items,
            batch_scores=batch_scores,
            purchased_by_user=purchased_by_user,
            k=k,
        )

        # Расчет NDCG@100
        for u in batch_users:
            item_idxs, _ = batch_recs[u]
            scores.append(
                ndcg_at_k(
                    item_idxs,
                    gt[u],
                    k,
                )
            )

    return float(np.mean(scores))

In [33]:
NDCG_100 = evaluate_ndcg(
    model=model,
    matrix=user_items_csr,
    users_for_eval=users_for_eval,
    gt=gt,
    purchased_by_user=purchased_by_user,
    k=100,
    batch_size=512,
    oversample=1000,
)

print(f"NDCG@100: {NDCG_100:.5f}")


100%|██████████| 663/663 [20:43<00:00,  1.87s/it]

NDCG@100: 0.01474


## Обучение модели на новом периоде и оценка на тесте

### Формироване матрицы **users × items**

In [34]:
NEW_TRAIN_START = date(2025, 5, 2)
NEW_TRAIN_END  = date(2025, 7, 1)

NEW_VAL_START = date(2025, 7, 2)
NEW_VAL_END  = date(2025, 7, 15)

# Директория для сохранения шардов
OUT_DIR = f"../data/processed/csr_build_{NEW_TRAIN_START}_{NEW_TRAIN_END}"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

In [35]:
tracker_ddf = load_interactions(train=True)
orders_ddf = load_orders(train=True)

build_interaction_csr(
    tracker_ddf=tracker_ddf,
    orders_ddf=orders_ddf,
    start_date=NEW_TRAIN_START,
    end_date=NEW_TRAIN_END,
    n_shards=N_SHARDS,
    out_dir=OUT_DIR,
)

Shard 1/128
Shard 2/128
Shard 3/128
Shard 4/128
Shard 5/128
Shard 6/128
Shard 7/128
Shard 8/128
Shard 9/128
Shard 10/128
Shard 11/128
Shard 12/128
Shard 13/128
Shard 14/128
Shard 15/128
Shard 16/128
Shard 17/128
Shard 18/128
Shard 19/128
Shard 20/128
Shard 21/128
Shard 22/128
Shard 23/128
Shard 24/128
Shard 25/128
Shard 26/128
Shard 27/128
Shard 28/128
Shard 29/128
Shard 30/128
Shard 31/128
Shard 32/128
Shard 33/128
Shard 34/128
Shard 35/128
Shard 36/128
Shard 37/128
Shard 38/128
Shard 39/128
Shard 40/128
Shard 41/128
Shard 42/128
Shard 43/128
Shard 44/128
Shard 45/128
Shard 46/128
Shard 47/128
Shard 48/128
Shard 49/128
Shard 50/128
Shard 51/128
Shard 52/128
Shard 53/128
Shard 54/128
Shard 55/128
Shard 56/128
Shard 57/128
Shard 58/128
Shard 59/128
Shard 60/128
Shard 61/128
Shard 62/128
Shard 63/128
Shard 64/128
Shard 65/128
Shard 66/128
Shard 67/128
Shard 68/128
Shard 69/128
Shard 70/128
Shard 71/128
Shard 72/128
Shard 73/128
Shard 74/128
Shard 75/128
Shard 76/128
Shard 77/128
Shard 78

In [36]:
user_items_csr = build_final_csr(
    out_dir=OUT_DIR,
    n_shards=N_SHARDS,
)

user_items_csr

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 272546528 stored elements and shape (951404, 4247685)>

### Обучение ALS

In [37]:
user_items_csr = load_npz(f"{OUT_DIR}/interaction_matrix.npz")
user_items_csr

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 272546528 stored elements and shape (951404, 4247685)>

In [38]:
model = AlternatingLeastSquares(
    factors=128,
    regularization=0.01,
    iterations=20,
    use_gpu=False,
)

model.fit(user_items_csr)

  0%|          | 0/20 [00:00<?, ?it/s]

In [39]:
model.save("../models/als_model.npz")

user_map = pd.read_parquet(f"{OUT_DIR}/user_mapping.parquet")
item_map = pd.read_parquet(f"{OUT_DIR}/item_mapping.parquet")

user_map.to_parquet("../models/als_user_mapping.parquet", index=False)
item_map.to_parquet("../models/als_item_mapping.parquet", index=False)

# Опционально сохраняем и под явными full-именами
user_map.to_parquet("../models/als_full_user_mapping.parquet", index=False)
item_map.to_parquet("../models/als_full_item_mapping.parquet", index=False)

print("Сохранено:")
print(" - ../models/als_model.npz")
print(" - ../models/als_user_mapping.parquet")
print(" - ../models/als_item_mapping.parquet")
print(" - ../models/als_full_user_mapping.parquet")
print(" - ../models/als_full_item_mapping.parquet")

Сохранено:
 - ../models/als_model.npz
 - ../models/als_user_mapping.parquet
 - ../models/als_item_mapping.parquet
 - ../models/als_full_user_mapping.parquet
 - ../models/als_full_item_mapping.parquet


In [40]:
orders_ddf_test = load_orders(train=False)
orders_ddf_test.head()

,item_id,user_id,created_timestamp,last_status,last_status_timestamp,created_date
0,7510328,981,2025-07-02 08:27:17.686,delivered_orders,2025-07-06 16:14:12,2025-07-02
1,136809690,4670,2025-07-02 17:17:30.423,canceled_orders,2025-07-06 19:35:07,2025-07-02
2,232878073,12341,2025-07-02 15:17:46.393,canceled_orders,2025-07-11 15:36:40,2025-07-02
3,42322137,12860,2025-07-02 00:40:52.503,delivered_orders,2025-07-04 19:44:48,2025-07-02
4,22480781,15330,2025-07-02 07:25:11.226,delivered_orders,2025-07-03 20:07:33,2025-07-02


In [41]:
orders_ddf = load_orders(train=True)

user_map = pd.read_parquet(f"{OUT_DIR}/user_mapping.parquet")
item_map = pd.read_parquet(f"{OUT_DIR}/item_mapping.parquet")

purchased_by_user = build_purchased_by_user_idx(
    orders_ddf, user_map, item_map, NEW_TRAIN_START, NEW_TRAIN_END
)

purchased_by_user

{np.int32(439877): array([   1457, 3243652, 2954297, 3766754,  208036,   80608], dtype=int32),
 np.int32(652383): array([   1457, 1498179, 1053211,  561259, 1976529, 2673634,  785800,
         205022, 3555204, 1850038, 1500186, 2251999, 4114528, 3159635,
        2205905,  388724, 1971881, 1181079,   41604, 1337295, 1992543,
        2121061], dtype=int32),
 np.int32(545580): array([   2379, 2125469, 4195138, 3020896, 3087449, 3414980, 1203624,
        3496404, 3064390,  418161, 1484589, 1178239, 2095747, 2040762,
        3086021, 3486830, 2596811, 1371866, 2016412, 2707292, 3661787,
        1835876,  795708, 2025405, 3559248, 2650500, 3763467, 3814855,
        1243483,  987719, 4091123, 4208315, 1865549, 1714531, 1724995,
        3093161, 2320376, 1979238,  709746,  325713,   95468, 2965682,
        3602894, 3786046,  473042, 2858889, 1016387, 1244984, 3384925,
        1187068, 2790964, 3458456], dtype=int32),
 np.int32(209013): array([   2379, 1054445,  647713, 4005055, 2813179, 217637

In [42]:
# ------------------------------------------------------------
# SAVE TEST CANDIDATES FOR SECOND LEVEL
# ------------------------------------------------------------
user_map = pd.read_parquet(f"{OUT_DIR}/user_mapping.parquet")
item_map = pd.read_parquet(f"{OUT_DIR}/item_mapping.parquet")

all_test_user_idxs = build_all_test_user_idxs(
    orders_ddf_test=orders_ddf_test,
    out_dir=OUT_DIR,
)

als_test_candidates = build_candidates_df(
    model=model,
    matrix=user_items_csr,
    user_idxs=all_test_user_idxs,
    purchased_by_user=purchased_by_user,
    user_map=user_map,
    item_map=item_map,
    k=200,
    oversample=1000,
    batch_size=512,
)

als_test_candidates.to_parquet("../data/interim/als_test_candidates.parquet", index=False)

print("Сохранено:")
print(" - ../data/interim/als_test_candidates.parquet")
als_test_candidates.head()

100%|██████████| 676/676 [40:36<00:00,  3.60s/it]


Сохранено:
 - ../data/interim/als_test_candidates.parquet


,user_id,item_id,als_score,als_rank,in_als
0,20920,222159914,0.258555,1,1
1,20920,245644634,0.253455,2,1
2,20920,136634977,0.240146,3,1
3,20920,15023057,0.237464,4,1
4,20920,51974017,0.222300,5,1


### Подготовка ground truth

In [43]:
eval_df, gt, users_for_eval = make_df_for_eval(
    orders_ddf_test, OUT_DIR, NEW_VAL_START, NEW_VAL_END
)

display(eval_df.head())
print("Users in val GT:", len(gt))
print("Avg GT size:", np.mean([len(v) for v in gt.values()]))

,user_id,item_id,u,i
0,981,7510328,184,93626
1,4670,136809690,887,1713853
2,12341,232878073,2323,2915706
3,12860,42322137,2422,530330
4,15330,22480781,2887,281708


Users in val GT: 345318
Avg GT size: 4.12001401606635


### Оценка на тестовой выборке

In [44]:
NDCG_100 = evaluate_ndcg(
    model=model,
    matrix=user_items_csr,
    users_for_eval=users_for_eval,
    gt=gt,
    purchased_by_user=purchased_by_user,
    k=100,
    batch_size=512,
    oversample=1000,
)

print(f"NDCG@100: {NDCG_100:.5f}")

100%|██████████| 675/675 [40:16<00:00,  3.58s/it]

NDCG@100: 0.01618
